In [1]:
import datetime
import os
import logging
from typing import Optional, Callable

from domain_adaptation_ct.dataset.image_dataset import TwoLabelDataset
from domain_adaptation_ct.learn.architectures import ResNet50DANN
from domain_adaptation_ct.learn.trainers import DANNTrainer
from domain_adaptation_ct.visualize.imshow_gray import ImShowGray

import numpy as np
import torch

In [2]:
model = ResNet50DANN.load("/data/D20_CV_V2/linear_increasing_lambda_scheduler_fold_0_2025-11-12_01-01-33/model.safetensors", num_classes=11, lamb_initial=0.0, ld_scale=1.0)
model.eval()
model.to('cuda')

ResNet50DANN(
  (resnet): ResNetModel(
    (embedder): ResNetEmbeddings(
      (embedder): ResNetConvLayer(
        (convolution): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
        (normalization): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (activation): ReLU()
      )
      (pooler): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
    )
    (encoder): ResNetEncoder(
      (stages): ModuleList(
        (0): ResNetStage(
          (layers): Sequential(
            (0): ResNetBottleNeckLayer(
              (shortcut): ResNetShortCut(
                (convolution): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
                (normalization): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
              )
              (layer): Sequential(
                (0): ResNetConvLayer(
                  (convolution): Conv2d(64, 64, kernel_siz

In [5]:
print(os.listdir("/data/DG_Test_set/"))
print(os.listdir("/data/D20_Folds/"))
print(os.listdir("/data/D21_Folds/"))

['full_test_dataset.npz', 'test_dataset_original.npz', 'test_dataset_ring_data.npz', 'test_dataset_Rotate_90deg.npz', 'test_dataset_Uniform_Noise.npz']
['fold0.npz', 'fold1.npz', 'fold2.npz', 'fold3.npz', 'fold4.npz']
['fold0.npz', 'fold1.npz', 'fold2.npz', 'fold3.npz', 'fold4.npz']


In [44]:
originals = TwoLabelDataset.load("/data/DG_Test_set/test_dataset_original.npz", convert_grayscale_to_rgb = True)
# THE EXPECTED RANGE IS [-1, 1] - NEED TO NORMALIZE ORIGINALS LIKEWISE. 

In [5]:
# rings = TwoLabelDataset.load("/data/DG_Test_set/test_dataset_ring_data.npz", convert_grayscale_to_rgb = True)

In [7]:
d20_fold0 = TwoLabelDataset.load("/data/D20_Folds/fold0.npz", convert_grayscale_to_rgb=True)

In [58]:

# Min-max normalize to range [-1, 1], under the assumption that we know the input data is from range [0, 1].

MIN = 0
MAX = 1


def normalize_image(image, mean=0.5, std=0.5):
    """
    Normalize an image tensor to have a mean and standard deviation.
    """
    return (image - mean) / std
 
def normalize_images(images, mean=0.5, std=0.5):
    """
    Normalize a list of images.
    """
    return [normalize_image(image, mean, std) for image in images]

def rescale_image_pixel_values(image, rescaled_min, rescaled_max):
    """
    Rescale a min-max normalized image's pixel values (assumed to be in range [0, 1]) to range [output_min, output_max].
    """
    scale_factor = rescaled_max - rescaled_min
    return (image * scale_factor) + rescaled_min

def rescale_images_pixel_values(images, rescaled_min, rescaled_max):
    """
    Rescale a list of min-max normalized images in [0, 1] to range [output_min, output_max].
    """
    return [rescale_image_pixel_values(image, rescaled_min, rescaled_max) for image in images]

# for i in range(100):
#     img = rescale_image_pixel_values(originals[i]['pixel_values'], rescaled_min = -1, rescaled_max = 1)
#     print(img.min(), img.max())

In [63]:
from sklearn.metrics import confusion_matrix

# Initialize lists for true and predicted labels
true_labels = []
pred_labels = []

# Loop over multiple indices
for idx in range(20):  # or specify a subset like range(100)
    instance = originals[idx]
    X = instance['pixel_values']
    X = rescale_image_pixel_values(X, rescaled_min = -1, rescaled_max = 1)
    X_cuda = X.unsqueeze(0).to('cuda')
    y = instance['labels1']  # true label
    y_cuda = torch.Tensor([y]).to(int).to('cuda')
    d = instance['labels2']  # optional if needed
    d_cuda = torch.Tensor([d]).to(int).to('cuda')

    print(X_cuda.min(), X_cuda.max())
    print(y_cuda)
    print(d_cuda)
    

    # ImShowGray.imshow(np.transpose(X, [1, 2, 0]), title=f"{y},{d}")

    # Move to GPU and run model
    outputs = model(X_cuda, y_cuda, d_cuda)
    del X_cuda
    del y_cuda
    del d_cuda

    print(outputs)
    
    # Get prediction
    pred = torch.argmax(outputs.branch1_logits).detach().cpu().item()

    # Store labels
    true_labels.append(y)
    pred_labels.append(pred)

# Compute confusion matrix
cm = confusion_matrix(true_labels, pred_labels)

print("Confusion Matrix:")
print(cm)

tensor(-0.7490, device='cuda:0') tensor(1., device='cuda:0')
tensor([4], device='cuda:0')
tensor([1], device='cuda:0')
BranchedOutput(loss=tensor(0.6977, device='cuda:0', grad_fn=<AddBackward0>), branch1_logits=tensor([[-0.3692, -4.6764, -2.1283,  0.0812,  7.0290,  1.1645, -4.3484, -2.0328,
         -3.7472,  4.1005,  6.7267]], device='cuda:0',
       grad_fn=<AddmmBackward0>), branch2_logits=tensor([[-0.0091]], device='cuda:0', grad_fn=<AddmmBackward0>), loss1=tensor(0., device='cuda:0', grad_fn=<MeanBackward0>), loss2=tensor(0.6977, device='cuda:0',
       grad_fn=<BinaryCrossEntropyWithLogitsBackward0>))
tensor(-0.7490, device='cuda:0') tensor(1., device='cuda:0')
tensor([0], device='cuda:0')
tensor([1], device='cuda:0')
BranchedOutput(loss=tensor(0.6906, device='cuda:0', grad_fn=<AddBackward0>), branch1_logits=tensor([[ 7.4524, -0.7696,  0.0307, -2.6188,  0.2616,  1.2998, -1.7137, -2.2157,
         -0.9811,  0.1651,  0.5303]], device='cuda:0',
       grad_fn=<AddmmBackward0>), bran